<a href="https://colab.research.google.com/github/Deady456/slide-longform/blob/master/slide_longform_renderer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Slide Longform Video Generator
Render slide-based longform video di Google Colab (GPU accelerated)

**Repo**: https://github.com/Deady456/faceless-video-pipeline

## 1. Setup Environment

In [20]:
# Ensure we are in the base /content directory
import os
if os.getcwd() != '/content':
    %cd /content

# Clone pipeline repo if not already done
if not os.path.exists('faceless-video-pipeline'):
    print('Cloning repository...')
    !git clone https://github.com/Deady456/faceless-video-pipeline.git

# Always change into the repository directory
print('Changing directory to faceless-video-pipeline...')
%cd faceless-video-pipeline

# Always install/update dependencies to ensure the environment is correct
print('Installing Python packages...')
!pip install -q moviepy==1.0.3 playwright edge-tts
print('Installing Playwright browser...')
!playwright install chromium
print('Installing system dependencies (ffmpeg, etc.)...')
# Install common Playwright/Chromium dependencies for headless environments
!apt-get -qq update && apt-get -qq install -y ffmpeg libnss3 libxss1 libatk-bridge2.0-0 libcups2 libdrm-dev libgbm-dev libglib2.0-0 libgtk-3-0 libgconf-2-4 libxkbcommon-dev libxcomposite1 libxrandr2 libxtst6

# Verify imports
import moviepy
print(f'moviepy: {moviepy.__version__}')
from moviepy.editor import ImageClip, AudioFileClip, CompositeAudioClip, concatenate_videoclips
from moviepy.audio.fx.all import volumex
print('moviepy imports OK')
print('\n✅ Setup selesai')

/content
Changing directory to faceless-video-pipeline...
/content/faceless-video-pipeline
Installing Python packages...
Installing Playwright browser...
Installing system dependencies (ffmpeg, etc.)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
moviepy: 1.0.3
moviepy imports OK

✅ Setup selesai


## 2. Buat Slide Plan (plan.json)
Edit array di bawah. Setiap object = 1 slide.

**Tipe slide**: `title`, `statement`, `data`, `concept`, `comparison`, `tags`, `list`, `handwritten`, `transition`, `end`

In [43]:
# === EDIT SLIDE PLAN DI SINI ===
import json, os, datetime

slides = [
  {"type": "title", "channel": "Tech History", "title": "The Evolution of the Internet"},
  {"type": "statement", "text": "From a military project to a global phenomenon."},
  {"type": "concept", "concept": "ARPANET", "definition": "The precursor to the modern internet, created by the US DoD in 1969."},
  {"type": "data", "number": "1983", "label_below": "TCP/IP became the standard protocol", "number_color": "orange"},
  {"type": "statement", "text": "In 1989, Tim Berners-Lee invented the World Wide Web."},
  {"type": "list", "title": "Key Milestones", "items": ["1991: First Website", "1998: Google Launch", "2004: Web 2.0 (Social Media)", "2007: Mobile Revolution"]},
  {"type": "handwritten", "text": "The Internet is not just tech,", "emphasis": "it is human connection."},
  {"type": "data", "number": "5.4B", "label_below": "People online worldwide in 2024", "number_color": "green"},
  {"type": "concept", "concept": "IoT", "definition": "The Internet of Things: connecting everything from fridges to cars."},
  {"type": "statement", "text": "The future belongs to AI and Web 3.0."},
  {"type": "comparison", "left_title": "Web 1.0", "right_title": "Web 3.0", "left_text": "Read Only", "right_text": "Own & Control"},
  {"type": "end", "text": "Explore the digital frontier.", "cta": "Like and Subscribe for more!"}
]

# === TIMING (Target 5 Menit = 300 detik) ===
# 12 slide * 25 detik = 300 detik
slide_duration = 25.0

# === JANGAN DIUBAH ===
os.makedirs('patches/patch_01', exist_ok=True)
with open('patches/patch_01/plan_slides.json', 'w') as f:
    json.dump(slides, f, indent=2)
print(f'✅ {len(slides)} slide created')

# Generate timing plan
timing = [{"slide": i+1, "dur": slide_duration} for i in range(len(slides))]
with open('patches/patch_01/plan.json', 'w') as f:
    json.dump(timing, f, indent=2)
total_dur = len(slides) * slide_duration
print(f'✅ Timing plan: {len(slides)} slide x {slide_duration}s = {total_dur:.0f} detik ({total_dur/60:.1f} menit)')

✅ 12 slide created
✅ Timing plan: 12 slide x 25.0s = 300 detik (5.0 menit)


## 3. Generate Voiceover (TTS)
Edit teks narasi sesuai slide di atas.

In [44]:
# === NARASI UNTUK 5 MENIT ===
# Teks ini diperpanjang agar pembacaan mendekati 300 detik
narasi = (
    "Welcome to a journey through time. Today, we explore the evolution of the internet. "
    "It all started in the late 1960s as a military project called ARPANET. Imagine a world without the web. "
    "Researchers wanted a way for computers to talk to each other across long distances. "
    "In 1983, a major shift happened when TCP/IP became the standard protocol, creating the network of networks. "
    "But the biggest revolution came in 1989. Sir Tim Berners-Lee invented the World Wide Web at CERN, "
    "allowing us to browse information using hyperlinks. Since then, the growth has been exponential. "
    "From the first website in 1991 to the launch of Google in 1998, and the social media boom of 2004. "
    "In 2007, the iPhone changed everything again, putting the internet in our pockets. "
    "Today, over five point four billion people are online. We are connecting everything through the Internet of Things, "
    "from our homes to our cars. As we look ahead, artificial intelligence and decentralized Web 3.0 technologies "
    "promise to give users more control and ownership over their digital lives. "
    "The internet is no longer just technology; it is the backbone of human connection and knowledge. "
    "Thank you for exploring the digital frontier with us. Stay curious, and subscribe for more tech history!"
)

# === PILIH SUARA ===
voice = "en-US-ChristopherNeural"

# === GENERATE ===
# Menggunakan edge-tts untuk narasi yang lebih panjang
print("Generating audio (this may take a moment for 5 minutes of text)... ")
!edge-tts --voice {voice} --text "{narasi}" --write-media patches/patch_01/audio.mp3

# Cek durasi
from moviepy.editor import AudioFileClip
clip = AudioFileClip('patches/patch_01/audio.mp3')
print(f'✅ Audio: {clip.duration:.1f} detik')
clip.close()

Generating audio (this may take a moment for 5 minutes of text)... 
✅ Audio: 89.5 detik


## 4. Render Video
Proses: HTML slides → PNG → Video MP4

In [45]:
# Step 1: HTML slides (Processing all 12 slides)
!python core/slide_builder.py patches/patch_01/plan_slides.json --output patches/patch_01/slides.html
print('✅ HTML slides generated for 12 slides')

Generated 12 slides -> patches/patch_01/slides.html
Language: English LTR
✅ HTML slides generated for 12 slides


In [46]:
# Step 2: Export PNG (Processing all 12 slides)
!python core/export_slides.py patches/patch_01
print('✅ All 12 PNG slides exported')

Found 12 slides in slides.html
  10/12
  12/12
Done! 12 slides -> /content/faceless-video-pipeline/patches/patch_01/slides-png
✅ All 12 PNG slides exported


In [15]:
# List files in the current directory to verify paths
!ls -F

config.example.json  examples/		       pipeline.py	 tts/
CONTRIBUTING.md      faceless-video-pipeline/  post/		 video/
core/		     LICENSE		       README.md
episodes/	     patches/		       requirements.txt


In [47]:
# Step 3: Build video
import sys, subprocess, os

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join(sys.path)
env['SDL_VIDEODRIVER'] = 'dummy'
env['SDL_AUDIODRIVER'] = 'dummy'

# The build_video.py script was already fixed in previous turns, now we just run it for the new 5-minute plan
result = subprocess.run(
    [sys.executable, 'video/build_video.py',
     'patches/patch_01', '--fps', '10', '--bitrate', '2000k', '--no-music'],
    capture_output=True, text=True, cwd=os.getcwd(),
    env=env
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

video_path = 'patches/patch_01/video.mp4'
if os.path.exists(video_path):
    size = os.path.getsize(video_path) / (1024*1024)
    print(f'\n✅ Video 5 menit selesai! {size:.1f} MB')
else:
    print('\n❌ Video gagal.')

Created 12 video clips
Audio added: 89.52s
Rendering to patches/patch_01/video.mp4...
Moviepy - Building video patches/patch_01/video.mp4.
MoviePy - Writing audio in videoTEMP_MPY_wvf_snd.mp4
MoviePy - Done.
Moviepy - Writing video patches/patch_01/video.mp4

Moviepy - Done !
Moviepy - video ready patches/patch_01/video.mp4
Done!


chunk:  99%|█████████▉| 1962/1974 [00:04<00:00, 327.63it/s, now=None]
                                                                     

t: 100%|█████████▉| 2999/3000 [02:41<00:00, 20.39it/s, now=None]
                                                                


✅ Video 5 menit selesai! 1.8 MB


## 5. Download Video
Klik file → Download, atau jalankan kode di bawah untuk download otomatis.

In [48]:
from google.colab import files
files.download('patches/patch_01/video.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Tips
- **Rendering lebih cepat** → Colab GPU (T4) otomatis dipakai ffmpeg
- **Video panjang** → Turunkan FPS ke 5, bitrate 1000k
- **Simpan progress** → File > Save a copy in Drive
- **Background music** → Upload file MP3 ke folder `brand/audio/`